# Aula 2 - Exercício: O Labirinto do Tesouro

Nesta aula, a turma vai completar um agente que aprende a atravessar um mapa com paredes, tesouros, lava, vento, chave e porta.

A ideia é transformar Q-learning em uma atividade mais investigativa:

- primeiro entender o ambiente;
- depois implementar exploração e atualização da tabela Q;
- por fim comparar políticas e ajustar recompensas/hiperparâmetros.

O notebook tem TODOs para estudantes e um gabarito no final para o professor.


## Regras do ambiente

Símbolos do mapa:

- `S`: início;
- `G`: objetivo final;
- `#`: parede;
- `T`: tesouro;
- `K`: chave;
- `D`: porta, só passa com a chave;
- `L`: lava, manda o agente de volta ao início;
- `W`: vento, empurra o agente uma casa para a direita se possível;
- `.`: caminho livre.

Recompensas:

- cada passo custa um pouco;
- bater em parede custa mais um pouco;
- pegar tesouro recompensa;
- pegar chave recompensa;
- cair na lava penaliza;
- chegar ao objetivo com a chave encerra o episódio com grande recompensa.


In [ ]:
import random
import math
from collections import defaultdict, deque

try:
    import matplotlib.pyplot as plt
    HAS_PLOTS = True
except Exception:
    HAS_PLOTS = False

SEED = 11
random.seed(SEED)

ACTIONS = {
    0: (-1, 0),  # cima
    1: (0, 1),   # direita
    2: (1, 0),   # baixo
    3: (0, -1),  # esquerda
}
ACTION_NAMES = {0: "↑", 1: "→", 2: "↓", 3: "←"}

def moving_average(values, window=100):
    if not values:
        return []
    out, total, fila = [], 0.0, deque()
    for value in values:
        total += value
        fila.append(value)
        if len(fila) > window:
            total -= fila.popleft()
        out.append(total / len(fila))
    return out

def plot_series(values, title="", ylabel="retorno", window=100):
    media = moving_average(values, window)
    if HAS_PLOTS:
        plt.figure(figsize=(10, 3))
        plt.plot(media)
        plt.title(title)
        plt.xlabel("episódio")
        plt.ylabel(f"{ylabel} médio ({window})")
        plt.grid(alpha=0.25)
        plt.show()
    else:
        inicio = sum(values[:window]) / max(1, min(window, len(values)))
        fim = sum(values[-window:]) / max(1, min(window, len(values)))
        print(title)
        print(f"episódios: {len(values)} | média inicial={inicio:.2f} | média final={fim:.2f} | melhor={max(values):.2f}")

def argmax_random_tie(values):
    best = max(values)
    options = [i for i, value in enumerate(values) if value == best]
    return random.choice(options)


In [ ]:
class TreasureMazeEnv:
    MAP = [
        "##########",
        "#S..T...K#",
        "#.##.##..#",
        "#....L...#",
        "#T.##.##D#",
        "#....W.#G#",
        "##########",
    ]

    def __init__(self):
        self.grid = [list(row) for row in self.MAP]
        self.height = len(self.grid)
        self.width = len(self.grid[0])
        self.n_actions = 4
        self.start = self._find("S")
        self.goal = self._find("G")
        self.key_pos = self._find("K")
        self.treasures = self._find_all("T")

    def _find(self, symbol):
        for r, row in enumerate(self.grid):
            for c, value in enumerate(row):
                if value == symbol:
                    return (r, c)
        raise ValueError(f"símbolo {symbol!r} não encontrado")

    def _find_all(self, symbol):
        positions = []
        for r, row in enumerate(self.grid):
            for c, value in enumerate(row):
                if value == symbol:
                    positions.append((r, c))
        return positions

    def reset(self):
        self.pos = self.start
        self.has_key = False
        self.treasure_mask = 0
        self.steps = 0
        return self._state()

    def _state(self):
        return (self.pos[0], self.pos[1], int(self.has_key), self.treasure_mask)

    def valid_actions(self):
        return range(self.n_actions)

    def _tile(self, pos):
        r, c = pos
        return self.grid[r][c]

    def _is_blocked(self, pos):
        r, c = pos
        if r < 0 or r >= self.height or c < 0 or c >= self.width:
            return True
        if self.grid[r][c] == "#":
            return True
        if self.grid[r][c] == "D" and not self.has_key:
            return True
        return False

    def step(self, action):
        self.steps += 1
        dr, dc = ACTIONS[action]
        r, c = self.pos
        candidate = (r + dr, c + dc)

        reward = -0.03
        done = False
        info = {}

        if self._is_blocked(candidate):
            reward -= 0.12
            candidate = self.pos
            info["evento"] = "bloqueado"

        self.pos = candidate
        tile = self._tile(self.pos)

        if tile == "W":
            pushed = (self.pos[0], self.pos[1] + 1)
            if not self._is_blocked(pushed):
                self.pos = pushed
                reward += 0.03
                info["evento"] = "vento"

        tile = self._tile(self.pos)

        if tile == "K" and not self.has_key:
            self.has_key = True
            reward += 0.80
            info["evento"] = "chave"

        if tile == "T":
            idx = self.treasures.index(self.pos)
            bit = 1 << idx
            if not (self.treasure_mask & bit):
                self.treasure_mask |= bit
                reward += 1.00
                info["evento"] = "tesouro"

        if tile == "L":
            reward -= 2.00
            self.pos = self.start
            info["evento"] = "lava"

        if tile == "G":
            if self.has_key:
                collected = bin(self.treasure_mask).count("1")
                reward += 4.00 + collected
                done = True
                info["evento"] = "objetivo"
            else:
                reward -= 0.50
                info["evento"] = "sem_chave"

        return self._state(), reward, done, info

    def render(self, Q=None):
        state = self._state()
        policy_action = None
        if Q is not None and state in Q:
            policy_action = argmax_random_tie(Q[state])

        for r, row in enumerate(self.grid):
            chars = []
            for c, tile in enumerate(row):
                pos = (r, c)
                ch = tile
                if pos == self.pos:
                    ch = "A" if policy_action is None else ACTION_NAMES[policy_action]
                elif tile == "T":
                    idx = self.treasures.index(pos)
                    if self.treasure_mask & (1 << idx):
                        ch = "."
                elif tile == "K" and self.has_key:
                    ch = "."
                chars.append(ch)
            print("".join(chars))
        print(f"estado={state} chave={self.has_key} passos={self.steps}\n")


env = TreasureMazeEnv()
state = env.reset()
env.render()


## Exercício 1 - Política epsilon-greedy

Complete a função abaixo.

Com probabilidade `epsilon`, ela deve escolher uma ação aleatória.

Caso contrário, deve escolher uma das ações de maior valor em `Q[state]`.


In [ ]:
def epsilon_greedy(Q, state, n_actions, epsilon):
    # TODO estudante:
    # 1. Se o estado ainda não existe em Q, o defaultdict já cria uma lista de zeros.
    # 2. Com probabilidade epsilon, escolha uma ação aleatória.
    # 3. Caso contrário, escolha a ação de maior valor, desempate aleatoriamente.
    raise NotImplementedError("implemente epsilon_greedy")


## Exercício 2 - Atualização Q-learning

Complete a função de atualização:

```text
alvo = reward + gamma * max_a Q[next_state][a]
Q[state][action] = Q[state][action] + alpha * (alvo - Q[state][action])
```

Se `done=True`, o futuro deve valer zero.


In [ ]:
def q_learning_update(Q, state, action, reward, next_state, done, alpha, gamma):
    # TODO estudante:
    # 1. Calcule o melhor valor futuro.
    # 2. Calcule o alvo.
    # 3. Atualize Q[state][action].
    # 4. Retorne Q[state][action].
    raise NotImplementedError("implemente q_learning_update")


## Exercício 3 - Treinamento

Agora complete o laço de treino.

Dicas:

- `epsilon` deve diminuir ao longo dos episódios;
- em cada passo, escolha ação, chame `env.step(action)` e atualize Q;
- guarde o retorno total de cada episódio em uma lista.


In [ ]:
def train_agent(env, episodes=3000, max_steps=160, alpha=0.20, gamma=0.96,
                epsilon_start=1.0, epsilon_end=0.05, seed=SEED):
    # TODO estudante:
    # 1. Crie Q como defaultdict(lambda: [0.0] * env.n_actions).
    # 2. Para cada episódio, resete o ambiente.
    # 3. Reduza epsilon linearmente de epsilon_start até epsilon_end.
    # 4. Rode até max_steps ou done=True.
    # 5. Retorne Q e a lista de retornos.
    raise NotImplementedError("implemente train_agent")


## Testes rápidos

Rode esta célula depois de implementar os três exercícios.


In [ ]:
try:
    teste_Q = defaultdict(lambda: [0.0] * 4)
    teste_Q["s"] = [0.0, 1.0, 1.0, -1.0]
    escolhas = {epsilon_greedy(teste_Q, "s", 4, epsilon=0.0) for _ in range(30)}
    assert escolhas <= {1, 2}, escolhas

    valor = q_learning_update(teste_Q, "s", 0, reward=1.0, next_state="s2",
                              done=True, alpha=0.5, gamma=0.9)
    assert abs(valor - 0.5) < 1e-9, valor

    print("testes passaram")
except NotImplementedError as exc:
    print(f"ainda falta: {exc}")


## Treine seu agente

Depois dos TODOs, esta célula deve treinar uma política para o labirinto.


In [ ]:
try:
    env = TreasureMazeEnv()
    Q, returns = train_agent(
        env,
        episodes=4500,
        max_steps=180,
        alpha=0.20,
        gamma=0.97,
        epsilon_start=1.0,
        epsilon_end=0.04,
    )
    plot_series(returns, "Labirinto do Tesouro", ylabel="retorno", window=200)
    print(f"estados visitados: {len(Q)}")
except NotImplementedError as exc:
    print(f"implemente antes de treinar: {exc}")


In [ ]:
def run_episode(env, Q, max_steps=80, render=True):
    state = env.reset()
    total = 0.0
    if render:
        env.render(Q)
    for step in range(max_steps):
        action = argmax_random_tie(Q[state])
        state, reward, done, info = env.step(action)
        total += reward
        if render:
            print(f"passo={step + 1:02d} ação={ACTION_NAMES[action]} reward={reward:.2f} info={info}")
            env.render(Q)
        if done:
            break
    print(f"retorno total: {total:.2f}")
    return total

try:
    run_episode(TreasureMazeEnv(), Q, max_steps=80, render=True)
except NameError:
    print("treine o agente primeiro para criar a variável Q")


## Desafios para a turma

1. **Leaderboard:** quem consegue maior retorno médio em 30 episódios gulosos?
2. **Reward shaping:** mude as recompensas. O agente passa a pegar tesouros ou corre direto para a saída?
3. **Ablation:** remova o vento `W`. O treinamento fica mais fácil?
4. **Exploração:** compare `epsilon_end=0.30` com `epsilon_end=0.01`.
5. **Algoritmo extra:** implemente SARSA e compare com Q-learning.
6. **Rede neural:** treine uma rede para imitar sua tabela Q e discuta o que ainda falta para virar DQN.


In [ ]:
def evaluate(Q, episodes=30, max_steps=100):
    scores = []
    successes = 0
    for _ in range(episodes):
        env = TreasureMazeEnv()
        state = env.reset()
        total = 0.0
        done = False
        for _ in range(max_steps):
            action = argmax_random_tie(Q[state])
            state, reward, done, info = env.step(action)
            total += reward
            if done:
                successes += 1
                break
        scores.append(total)
    avg = sum(scores) / len(scores)
    print(f"retorno médio: {avg:.2f} | sucessos: {successes}/{episodes} | melhor: {max(scores):.2f}")
    return avg, successes

try:
    evaluate(Q)
except NameError:
    print("treine o agente primeiro para criar a variável Q")


## Gabarito do professor

A célula abaixo substitui as funções TODO por uma implementação completa.

Sugestão: deixe esta parte no seu notebook de professor e remova/oculte antes de distribuir aos alunos.


In [ ]:
def epsilon_greedy(Q, state, n_actions, epsilon):
    if random.random() < epsilon:
        return random.randrange(n_actions)
    return argmax_random_tie(Q[state])

def q_learning_update(Q, state, action, reward, next_state, done, alpha, gamma):
    future = 0.0 if done else max(Q[next_state])
    target = reward + gamma * future
    Q[state][action] += alpha * (target - Q[state][action])
    return Q[state][action]

def train_agent(env, episodes=3000, max_steps=160, alpha=0.20, gamma=0.96,
                epsilon_start=1.0, epsilon_end=0.05, seed=SEED):
    random.seed(seed)
    Q = defaultdict(lambda: [0.0] * env.n_actions)
    returns = []

    for ep in range(episodes):
        frac = ep / max(1, episodes - 1)
        epsilon = epsilon_start + frac * (epsilon_end - epsilon_start)
        state = env.reset()
        total = 0.0

        for _ in range(max_steps):
            action = epsilon_greedy(Q, state, env.n_actions, epsilon)
            next_state, reward, done, info = env.step(action)
            q_learning_update(Q, state, action, reward, next_state, done, alpha, gamma)
            state = next_state
            total += reward
            if done:
                break

        returns.append(total)

    return Q, returns

print("gabarito carregado")


In [ ]:
env = TreasureMazeEnv()
Q, returns = train_agent(
    env,
    episodes=4500,
    max_steps=180,
    alpha=0.20,
    gamma=0.97,
    epsilon_start=1.0,
    epsilon_end=0.04,
)
plot_series(returns, "Labirinto do Tesouro - gabarito", ylabel="retorno", window=200)
evaluate(Q)


## Extensão opcional - Rede neural e política

Esta extensão é uma ponte para **Deep Q-Networks**.

A rede abaixo recebe o estado do labirinto e tenta imitar a ação escolhida pela tabela Q:

```text
rede_neural(estado) -> preferência por [cima, direita, baixo, esquerda]
```

Aqui ela aprende imitando a política gulosa da tabela `Q` já treinada. Em um DQN completo, a rede preveria valores Q e aprenderia diretamente das transições `(estado, ação, recompensa, próximo_estado)`, usando replay buffer e uma rede alvo.


In [ ]:
class TinyMazePolicyNetwork:
    def __init__(self, n_inputs, n_hidden, n_outputs, lr=0.015, seed=SEED):
        random.seed(seed)
        self.lr = lr
        self.w1 = [[random.uniform(-0.35, 0.35) for _ in range(n_inputs)] for _ in range(n_hidden)]
        self.b1 = [0.0 for _ in range(n_hidden)]
        self.w2 = [[random.uniform(-0.35, 0.35) for _ in range(n_hidden)] for _ in range(n_outputs)]
        self.b2 = [0.0 for _ in range(n_outputs)]

    def forward(self, x):
        hidden = []
        for h, weights in enumerate(self.w1):
            z = self.b1[h] + sum(w * xi for w, xi in zip(weights, x))
            hidden.append(math.tanh(z))

        out = []
        for o, weights in enumerate(self.w2):
            y = self.b2[o] + sum(w * hi for w, hi in zip(weights, hidden))
            out.append(y)
        return hidden, out

    def predict(self, x):
        return self.forward(x)[1]

    def train_one(self, x, target):
        hidden, out = self.forward(x)
        grad_out = [2 * (out[o] - target[o]) / len(out) for o in range(len(out))]
        old_w2 = [row[:] for row in self.w2]

        for o in range(len(out)):
            for h in range(len(hidden)):
                self.w2[o][h] -= self.lr * grad_out[o] * hidden[h]
            self.b2[o] -= self.lr * grad_out[o]

        for h in range(len(hidden)):
            grad_h = sum(grad_out[o] * old_w2[o][h] for o in range(len(out)))
            grad_h *= 1 - hidden[h] ** 2
            for i in range(len(x)):
                self.w1[h][i] -= self.lr * grad_h * x[i]
            self.b1[h] -= self.lr * grad_h

        return sum((out[o] - target[o]) ** 2 for o in range(len(out))) / len(out)


def encode_maze_state(state, env):
    row, col, has_key, treasure_mask = state
    features = [-1.0 for _ in range(env.height * env.width)]
    features[row * env.width + col] = 1.0
    features.append(has_key * 2 - 1)
    for idx in range(len(env.treasures)):
        features.append(1.0 if treasure_mask & (1 << idx) else -1.0)
    return features


def greedy_actions_from_q_values(q_values):
    best = max(q_values)
    return [action for action, value in enumerate(q_values) if value == best]


def train_neural_maze_policy(Q, epochs=90):
    env = TreasureMazeEnv()
    dataset = []
    for state, q_values in Q.items():
        if any(abs(value) > 1e-9 for value in q_values):
            target = [-1.0 for _ in range(env.n_actions)]
            best_actions = greedy_actions_from_q_values(q_values)
            for action in best_actions:
                target[action] = 1.0
            dataset.append((encode_maze_state(state, env), target, best_actions))

    net = TinyMazePolicyNetwork(
        n_inputs=env.height * env.width + 1 + len(env.treasures),
        n_hidden=16,
        n_outputs=env.n_actions,
        lr=0.025,
    )
    losses = []
    for _ in range(epochs):
        random.shuffle(dataset)
        total_loss = 0.0
        for x, target, _ in dataset:
            total_loss += net.train_one(x, target)
        losses.append(total_loss / max(1, len(dataset)))

    correct = 0
    for x, _, best_actions in dataset:
        predicted = argmax_random_tie(net.predict(x))
        if predicted in best_actions:
            correct += 1

    print(f"amostras da tabela Q: {len(dataset)}")
    print(f"loss inicial: {losses[0]:.4f} | loss final: {losses[-1]:.4f}")
    print(f"acurácia imitando a ação gulosa: {correct / max(1, len(dataset)):.1%}")
    return net, losses


def evaluate_neural_policy(net, episodes=30, max_steps=100):
    scores = []
    successes = 0
    for _ in range(episodes):
        env = TreasureMazeEnv()
        state = env.reset()
        total = 0.0
        for _ in range(max_steps):
            values = net.predict(encode_maze_state(state, env))
            action = argmax_random_tie(values)
            state, reward, done, info = env.step(action)
            total += reward
            if done:
                successes += 1
                break
        scores.append(total)
    avg = sum(scores) / len(scores)
    print(f"rede neural | retorno médio: {avg:.2f} | sucessos: {successes}/{episodes} | melhor: {max(scores):.2f}")
    return avg, successes


maze_net, nn_losses = train_neural_maze_policy(Q)
evaluate_neural_policy(maze_net)
plot_series(nn_losses, "Rede neural imitando a política Q no labirinto", ylabel="loss", window=5)
